<a href="https://colab.research.google.com/github/PratamaAngga/Data-Mining-Course/blob/main/Diabetes-dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# SOAL 3 & 4 - Dataset Diabetes
# ============================================================

print("\n" + "=" * 60)
print("SOAL 3 & 4 - DATASET DIABETES")
print("=" * 60)

# ----------------------------------------------------------
# SOAL 3: Download Dataset Diabetes dari Kaggle
# CARA DOWNLOAD DI GOOGLE COLAB:
# ----------------------------------------------------------
# Opsi 1 (Kaggle API):
# from google.colab import files
# files.upload()  # upload kaggle.json
# !mkdir -p ~/.kaggle
# !cp kaggle.json ~/.kaggle/
# !chmod 600 ~/.kaggle/kaggle.json
# !kaggle datasets download -d mathchi/diabetes-data-set
# !unzip diabetes-data-set.zip

# Opsi 2 (langsung dari URL publik - LEBIH MUDAH):
diabetes_url = "https://raw.githubusercontent.com/plotly/datasets/master/diabetes.csv"
try:
    diabetes_raw = pd.read_csv(diabetes_url)
    print("✅ Dataset Diabetes berhasil diload!")
    print(f"Shape: {diabetes_raw.shape}")
    print(f"\nKolom: {diabetes_raw.columns.tolist()}")
except:
    print("⚠️ Gagal load dari URL. Gunakan Opsi Kaggle API di atas.")

# ----------------------------------------------------------
# SOAL 3: Memahami Dataset
# ----------------------------------------------------------
print("\n--- INFO DATASET DIABETES ---")
print(diabetes_raw.info())
print("\n--- STATISTIK DESKRIPTIF ---")
print(diabetes_raw.describe())

# ----------------------------------------------------------
# SOAL 4: Fitur bernilai 0 = Missing Values
# Kolom yang TIDAK BOLEH bernilai 0 secara medis:
# Glucose, BloodPressure, SkinThickness, Insulin, BMI
# (Kecuali Pregnancies dan DiabetesPedigreeFunction dan Age)
# ----------------------------------------------------------
print("\n--- SOAL 4: CEK NILAI 0 YANG MERUPAKAN MISSING VALUES ---")

cols_no_zero = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']

print("\nJumlah nilai 0 per kolom (yang seharusnya tidak boleh 0):")
for col in cols_no_zero:
    n_zero = (diabetes_raw[col] == 0).sum()
    pct    = n_zero / len(diabetes_raw) * 100
    print(f"  {col:20s}: {n_zero:3d} ({pct:.1f}%)")

# Ganti nilai 0 dengan NaN di kolom tersebut
diabetes = diabetes_raw.copy()
diabetes[cols_no_zero] = diabetes[cols_no_zero].replace(0, np.nan)

print("\nSetelah replace 0 → NaN:")
print(diabetes.isnull().sum())

# ----------------------------------------------------------
# SOAL 4: Imputasi Missing Values
# ----------------------------------------------------------
print("\n--- IMPUTASI MISSING VALUES ---")

# Strategi: isi dengan median per kelompok Outcome
# (lebih informatif dibanding median global)
for col in cols_no_zero:
    median_by_outcome = diabetes.groupby('Outcome')[col].transform('median')
    diabetes[col].fillna(median_by_outcome, inplace=True)

print("Setelah imputasi:")
print(diabetes.isnull().sum())
print("\nStatistik setelah imputasi:")
print(diabetes.describe())

# ----------------------------------------------------------
# SOAL 3 & 4: Visualisasi Dataset Diabetes
# ----------------------------------------------------------
print("\n--- VISUALISASI DATASET DIABETES ---")

# Distribusi Outcome
plt.figure(figsize=(6, 5))
sns.countplot(x='Outcome', data=diabetes, palette=['#3498db', '#e74c3c'])
plt.title('Soal 3 - Distribusi Outcome Diabetes', fontweight='bold')
plt.xticks([0, 1], ['Tidak Diabetes', 'Diabetes'])
plt.ylabel('Jumlah Pasien')
for p in plt.gca().patches:
    plt.gca().annotate(f'{int(p.get_height())}',
                       (p.get_x() + p.get_width()/2., p.get_height()),
                       ha='center', va='bottom', fontsize=12)
plt.tight_layout()
plt.savefig('soal3_outcome_dist.png', dpi=120, bbox_inches='tight')
plt.show()

# Distribusi semua fitur sebelum vs sesudah imputasi
fig, axes = plt.subplots(2, 5, figsize=(20, 8))
feature_cols = ['Pregnancies', 'Glucose', 'BloodPressure',
                'SkinThickness', 'Insulin', 'BMI',
                'DiabetesPedigreeFunction', 'Age']

for i, col in enumerate(feature_cols):
    row, c = divmod(i, 5)
    if row < 2:
        # Histogram sebelum vs sesudah dibedakan warna
        ax = axes[row][c] if i < 5 else axes[1][i-5]
        diabetes[col].hist(ax=ax, bins=20, color='#3498db', alpha=0.7, edgecolor='white')
        ax.set_title(col, fontweight='bold', fontsize=10)
        ax.set_xlabel('Value')
        ax.set_ylabel('Count')
        ax.grid(alpha=0.3)

# Hapus subplot kosong jika ada
for j in range(len(feature_cols), 10):
    row, c = divmod(j, 5)
    if row < 2:
        axes[row][c].set_visible(False)

plt.suptitle('Soal 3 - Distribusi Fitur Diabetes (Setelah Imputasi)',
             y=1.02, fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('soal3_feature_distribution.png', dpi=120, bbox_inches='tight')
plt.show()

# Boxplot per fitur by Outcome
fig, axes = plt.subplots(2, 4, figsize=(18, 10))
feature_cols = ['Pregnancies', 'Glucose', 'BloodPressure',
                'SkinThickness', 'Insulin', 'BMI',
                'DiabetesPedigreeFunction', 'Age']

for i, col in enumerate(feature_cols):
    row, c = divmod(i, 4)
    sns.boxplot(x='Outcome', y=col, data=diabetes,
                palette=['#3498db', '#e74c3c'], ax=axes[row][c])
    axes[row][c].set_title(f'{col} by Outcome', fontweight='bold')
    axes[row][c].set_xticks([0, 1])
    axes[row][c].set_xticklabels(['Tidak Diabetes', 'Diabetes'])

plt.suptitle('Soal 3 - Boxplot Fitur vs Outcome Diabetes',
             y=1.02, fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('soal3_boxplot_by_outcome.png', dpi=120, bbox_inches='tight')
plt.show()

# Heatmap Korelasi Diabetes
plt.figure(figsize=(10, 8))
colormap = sns.diverging_palette(220, 10, as_cmap=True)
sns.heatmap(diabetes.corr(), annot=True, fmt='.2f', cmap=colormap,
            square=True, linewidths=0.5, vmax=1.0, vmin=-1.0)
plt.title('Soal 3 - Pearson Correlation Heatmap (Diabetes)',
          fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('soal3_heatmap_diabetes.png', dpi=120, bbox_inches='tight')
plt.show()

# KDE Plot Glucose dan BMI by Outcome
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, col in zip(axes, ['Glucose', 'BMI']):
    for outcome, color, label in zip([0, 1], ['#3498db', '#e74c3c'],
                                     ['Tidak Diabetes', 'Diabetes']):
        subset = diabetes[diabetes['Outcome'] == outcome][col]
        sns.kdeplot(subset, ax=ax, fill=True, color=color, alpha=0.4, label=label)
    ax.set_title(f'Soal 3 - KDE {col} by Outcome', fontweight='bold')
    ax.set_xlabel(col)
    ax.legend()

plt.tight_layout()
plt.savefig('soal3_kdeplot_diabetes.png', dpi=120, bbox_inches='tight')
plt.show()

# Pairplot (key features)
key_features = ['Glucose', 'BMI', 'Age', 'Insulin', 'Outcome']
pp = sns.pairplot(diabetes[key_features], hue='Outcome',
                  palette=['#3498db', '#e74c3c'], plot_kws={'alpha': 0.5})
pp.fig.suptitle('Soal 3 - Pairplot Key Features Diabetes', y=1.02,
                fontsize=13, fontweight='bold')
plt.savefig('soal3_pairplot.png', dpi=120, bbox_inches='tight')
plt.show()

# ----------------------------------------------------------
# SOAL 4: Analisa Kesimpulan
# ----------------------------------------------------------
print("\n--- KESIMPULAN SOAL 4: PENANGANAN MISSING VALUES DIABETES ---")
print("""
Kolom yang memiliki nilai 0 sebagai missing values (secara medis tidak mungkin bernilai 0):
  - Glucose       : kadar gula darah tidak mungkin 0
  - BloodPressure : tekanan darah tidak mungkin 0
  - SkinThickness : ketebalan kulit tidak mungkin 0
  - Insulin       : kadar insulin tidak mungkin 0
  - BMI           : indeks massa tubuh tidak mungkin 0

Metode Imputasi yang digunakan:
  → Median per kelompok Outcome (lebih representatif daripada median global)
  → Alasan: pasien diabetes dan non-diabetes memiliki distribusi nilai yang berbeda,
    sehingga mengisi dengan median per kelompok lebih akurat.

Kolom Pregnancies TIDAK diganti karena nilai 0 valid (belum pernah hamil).
""")